In [1]:
import pandas as pd

In [3]:
df = pd.read_excel("data/05. Database RP May 2025 - AC REGISTER.xlsx", sheet_name="Raw", skiprows=1)

In [4]:
df = df.iloc[:, 1:]
df.head()

,YEAR,PERIODE,AIRCRAFT TYPE,FLIGHT NUMBER_CITYPAIR,SERVICE TYPE,SUB-SERVICE,ROUNDTRIPROUTE,FLIGHT ROUTE,AC REG,DATE,...,RPK (000) C CLASS,RPK (000) Y CLASS,OTHER REVENUE PASSENGER,OTHER REVENUE FREIGHT,QUARTER,STATUS,AIRCRAFT TYPE GROUPING,FLIGHT TYPE,GA Service,Region
0,2024,JAN,AC738,GA0072.CGK-TKG.[CGK-TKG],DOM,DOM_2 (JKT),CGK-TKG-CGK,CGK-TKG,PKGNR,30012024,...,0.573,28.268,382.164660,1.366988,Q1,ACTUAL,B738NG,PAX & CARGO,IBB,DOM_WEST
1,2024,JAN,AC738,GA0072.CGK-TKG.[CGK-TKG],DOM,DOM_2 (JKT),CGK-TKG-CGK,CGK-TKG,PKGFJ,31012024,...,0.955,27.886,427.183511,15.077904,Q1,ACTUAL,B738NG,PAX & CARGO,IBB,DOM_WEST
2,2024,JAN,AC738,GA0072.CGK-TKG.[CGK-TKG],DOM,DOM_2 (JKT),CGK-TKG-CGK,CGK-TKG,PKGNA,1012024,...,0.191,28.841,489.566409,47.838249,Q1,ACTUAL,B738NG,PAX & CARGO,IBB,DOM_WEST
3,2024,JAN,AC738,GA0072.CGK-TKG.[CGK-TKG],DOM,DOM_2 (JKT),CGK-TKG-CGK,CGK-TKG,PKGNR,2012024,...,2.483,27.886,593.068566,1.376459,Q1,ACTUAL,B738NG,PAX & CARGO,IBB,DOM_WEST
4,2024,JAN,AC738,GA0072.CGK-TKG.[CGK-TKG],DOM,DOM_2 (JKT),CGK-TKG-CGK,CGK-TKG,PKGMW,19012024,...,0.573,28.077,448.429472,30.733542,Q1,ACTUAL,B738NG,PAX & CARGO,IBB,DOM_WEST


## Finding Correlation

In [6]:
df_num = df.select_dtypes(include=["number"])
corr = df_num.corrwith(df_num['FUEL BURN (IN LITER)']).sort_values(ascending=False).dropna()
corr.head(30)

/Users/liaristiana/opt/anaconda3/lib/python3.11/site-packages/numpy/lib/function_base.py:2854: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/liaristiana/opt/anaconda3/lib/python3.11/site-packages/numpy/lib/function_base.py:2855: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


FUEL BURN (IN LITER)                 1.000000
ATK (000)                            0.994556
FUEL AIRCRAFT                        0.992278
TOTAL DIRECT,INDIRECT,FLEET COSTS    0.990710
TOTAL DIRECT FLIGHT COSTS            0.990159
TOTAL BO COSTS                       0.989569
TOTAL COSTS                          0.989480
TOTAL DIRECT AND INDIRECT COSTS      0.988514
TOTAL DIRECT COSTS                   0.988046
ATK PASSENGER (000)                  0.985845
ASK (000)                            0.985670
ASK (000) Y CLASS                    0.979844
TOTAL FLEET COST                     0.974822
CABIN CREW TRAVEL                    0.973903
LEASE AIRCRAFT                       0.973803
CABIN CREW PERSON                    0.961062
ASK (000) C CLASS                    0.960013
MAINTENANCE RESERVE                  0.958679
TOTAL INDIRECT COSTS                 0.952993
FLIGHT KILOMETERS                    0.949363
RTK (000)                            0.945204
FLIGHT HOURS                      

In [7]:
corr.tail(10)

DATE                                 -0.001975
OTHER REVENUE FREIGHT                -0.016536
PASSENGER REVENUE DISCOUNT           -0.025875
CLF (%)                              -0.027122
PASSENGER REVENUE DISCOUNT Y CLASS   -0.028871
SLF (%)                              -0.035823
FREIGHT COMMISSION                   -0.049396
ROUTE RESULT 1                       -0.093977
ROUTE RESULT 2                       -0.160063
LOAD FACTOR (%)                      -0.190665
dtype: float64

## Remove zero

In [24]:
df[df['BLOCK HOURS']==0][['FLIGHT HOURS', 'BLOCK HOURS', 'FUEL BURN (IN LITER)', 'FUEL AIRCRAFT']]

,FLIGHT HOURS,BLOCK HOURS,FUEL BURN (IN LITER),FUEL AIRCRAFT
2166,1.000000e-08,0.0,0.0,0.0
2169,1.000000e-08,0.0,0.0,0.0
2172,1.000000e-08,0.0,0.0,0.0
2175,1.000000e-08,0.0,0.0,0.0
2178,1.000000e-08,0.0,0.0,0.0
...,...,...,...,...
106874,0.000000e+00,0.0,0.0,0.0
106875,0.000000e+00,0.0,0.0,0.0
107617,0.000000e+00,0.0,0.0,0.0
108932,0.000000e+00,0.0,0.0,0.0


In [21]:
df1 = df[(df['FUEL BURN (IN LITER)']!=0) & (df['FLIGHT HOURS']!=0)].copy()

### One Hot Encoding for categorical data

In [13]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder

In [15]:
selected_features = [
                    'ROUNDTRIPROUTE', 'AIRCRAFT TYPE', 'AC REG', 'SERVICE TYPE', 'FLIGHT ROUTE', # categorical
                    'CARGO CARRIED', 'FREIGHT CARRIED', 'ASK (000)', 'ATK (000)', 'ATK PASSENGER (000)', 
                    'ASK (000) Y CLASS', 'ASK (000) C CLASS', 'RTK (000)', 'RPK (000)', 'RPK (000) Y CLASS',
                    'RTK PASSENGER (000)', 'ADMINISTRATION HO'
                    ]

In [26]:
X = df1[selected_features].copy()
y = df1['FUEL BURN (IN LITER)'].copy()

In [28]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

from xgboost import XGBRegressor

### split the data set into training and testing

In [31]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [33]:
categorical_cols = ['ROUNDTRIPROUTE', 'AIRCRAFT TYPE', 'AC REG', 'SERVICE TYPE', 'FLIGHT ROUTE']
numeric_cols = list(set(selected_features) - set(categorical_cols))
numeric_cols

['ATK (000)',
 'RPK (000)',
 'ASK (000)',
 'RTK (000)',
 'RPK (000) Y CLASS',
 'RTK PASSENGER (000)',
 'CARGO CARRIED',
 'FREIGHT CARRIED',
 'ASK (000) C CLASS',
 'ATK PASSENGER (000)',
 'ASK (000) Y CLASS',
 'ADMINISTRATION HO']

### encode the categorical columns

In [37]:
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
encoder.fit(X_train[categorical_cols])

OneHotEncoder(handle_unknown='ignore', sparse_output=False)

In [39]:
X_train_encoded = encoder.transform(X_train[categorical_cols])
X_test_encoded = encoder.transform(X_test[categorical_cols])

In [41]:
encoded_cols = encoder.get_feature_names_out(categorical_cols)

X_train_encoded = pd.DataFrame(X_train_encoded, columns=encoded_cols, index=X_train.index)
X_test_encoded = pd.DataFrame(X_test_encoded, columns=encoded_cols, index=X_test.index)

In [43]:
X_train_final = pd.concat([X_train[numeric_cols], X_train_encoded], axis=1)
X_test_final = pd.concat([X_test[numeric_cols], X_test_encoded], axis=1)

In [93]:
X_test_final

,ATK (000),RPK (000),ASK (000),RTK (000),RPK (000) Y CLASS,RTK PASSENGER (000),CARGO CARRIED,FREIGHT CARRIED,ASK (000) C CLASS,ATK PASSENGER (000),...,FLIGHT ROUTE_TYO-DPS,FLIGHT ROUTE_TYO-MDC-DPS,FLIGHT ROUTE_UPG-BPN,FLIGHT ROUTE_UPG-CGK,FLIGHT ROUTE_UPG-DPS,FLIGHT ROUTE_UPG-KDI,FLIGHT ROUTE_UPG-MDC,FLIGHT ROUTE_UPG-TTE,FLIGHT ROUTE_YIA-CGK,FLIGHT ROUTE_YIA-DPS
81409,31.794462,110.154,270.378,11.375107,105.147,11.015391,214.0,214.0,20.028,27.030731,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
30410,13.868391,53.144,117.936,5.180084,52.416,5.154968,11.0,11.0,8.736,11.436522,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
53021,18.726133,157.280,159.246,15.295155,146.467,15.256166,0.0,0.0,11.796,15.443133,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
97035,16.744962,118.665,142.398,11.990357,112.512,11.866513,100.0,100.0,10.548,14.236308,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2783,6.381742,26.800,54.270,2.604247,26.800,2.599605,0.0,0.0,4.020,5.263290,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72216,8.077200,70.808,68.688,7.052311,67.416,6.868375,372.0,372.0,5.088,6.661233,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
43887,13.868385,60.424,117.936,5.873353,57.512,5.861114,0.0,0.0,8.736,11.436808,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10168,45.567586,255.944,387.504,26.815023,251.160,24.826569,810.5,810.5,28.704,37.581379,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
61527,228.337364,1196.088,1316.746,131.127073,1022.970,119.608781,2189.0,2189.0,188.856,131.642545,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## XGBoost

In [46]:
bst = XGBRegressor(n_estimators=500, max_depth=15, learning_rate=0.05, gamma=0.01, objective="reg:squarederror")
bst.fit(X_train_final, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=0.01, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=15,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, ...)

In [47]:
y_pred = bst.predict(X_test_final)

In [50]:
mean_absolute_percentage_error(y_test, y_pred)

0.0466485466880655

In [52]:
y_test[0:5]

81409    7696.0
30410    4177.0
53021    5823.0
97035    5658.0
2783     2658.0
Name: FUEL BURN (IN LITER), dtype: float64

In [54]:
y_pred[0:5]

array([8264.322 , 4033.473 , 5816.67  , 5441.411 , 2646.8408],
      dtype=float32)

## How to use in one single data

In [131]:
datum = X_test.iloc[[10]]
datum

,ROUNDTRIPROUTE,AIRCRAFT TYPE,AC REG,SERVICE TYPE,FLIGHT ROUTE,CARGO CARRIED,FREIGHT CARRIED,ASK (000),ATK (000),ATK PASSENGER (000),ASK (000) Y CLASS,ASK (000) C CLASS,RTK (000),RPK (000),RPK (000) Y CLASS,RTK PASSENGER (000),ADMINISTRATION HO
10691,DPS-MDC-TYO-MDC-DPS,AC330,PKGPV,INT,DPS-MDC-TYO,2244.0,2244.0,1024.08,177.586,102.3904,877.2,146.88,18.965052,97.92,97.92,9.792,5112.152389


In [135]:
datum_encoded = encoder.transform(datum[categorical_cols])
datum_encoded = pd.DataFrame(datum_encoded, columns=encoded_cols, index=datum.index)
datum_final = pd.concat([datum[numeric_cols], datum_encoded], axis=1)
datum_final

,ATK (000),RPK (000),ASK (000),RTK (000),RPK (000) Y CLASS,RTK PASSENGER (000),CARGO CARRIED,FREIGHT CARRIED,ASK (000) C CLASS,ATK PASSENGER (000),...,FLIGHT ROUTE_TYO-DPS,FLIGHT ROUTE_TYO-MDC-DPS,FLIGHT ROUTE_UPG-BPN,FLIGHT ROUTE_UPG-CGK,FLIGHT ROUTE_UPG-DPS,FLIGHT ROUTE_UPG-KDI,FLIGHT ROUTE_UPG-MDC,FLIGHT ROUTE_UPG-TTE,FLIGHT ROUTE_YIA-CGK,FLIGHT ROUTE_YIA-DPS
10691,177.586,97.92,1024.08,18.965052,97.92,9.792,2244.0,2244.0,146.88,102.3904,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### predicted value

In [139]:
bst.predict(datum_final)

array([33643.797], dtype=float32)

### actual value

In [141]:
y_test.iloc[10]

34266.0